# Analog Contact Counting (Python/MDAnalysis equivalent of densityProfiles-aa-cutoffs.vmd)

Produces `analog_contacts.dat` with columns:
`frame  resid  x  y  z  4.5A_cutoff`

- `x, y, z` = center of mass coordinates of the analog residue
- `*A_cutoff` = number of other analog residues with at least one heavy atom within that cutoff (0 = isolated)

Replace `PSFFILE`, `DCDFILE`, `AAA` with your values (same placeholders as the VMD script).

## 1. Parameters

In [ ]:
import os
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import polars as pl
import numpy as np
from pathlib import Path

# ─── User parameters ────────────────────────────────────────────────────
resname   = [ 
    "SCA", "SCV", "SCL", "SCI", "SCC", "SCM", "SCS", "SCT", "SCN", "SCQ",
    "SCF", "SCY", "SCW", "SCP", "GLYD",
    "SCHE", "SCHD", "SCDN", "SCEN", "SCKN", "SCRN",
    "SCHP", "SCD", "SCE", "SCCM", "SCYM", "SCK", "SCR",
    "SCRN-1", "SCW-1"
    ]                # Residue name of the analog
cutoffs  = [4.5]  # Contact cutoff distances (Å)

def preprocess(resname):
    !cat get_trajectory.sh | sed s=BBB={resname}= > temp.sh
    !bash temp.sh
    psf_file = f"./analog-{resname}.psf"        # PSF topology file
    output_file = f"{resname}_contacts.dat" # Output file name
    return psf_file, output_file
# ────────────────────────────────────────────────────────────────────────

## 2. Load trajectory and classify residues

In [ ]:
def load_cylinder_com(cyl_file):
    """Return {(frame, resid): z_cyl} from a cylinder_com table written by center.vmd.

    The table (frame, resid, z_cyl) holds the Nitschke cylinder-based *local*
    membrane COM z for each solute in each frame, already expressed in the
    globally-centered frame. Returns {} if the file is missing (in which case the
    reaction coordinate falls back to the global all-lipid membrane COM)."""
    if not cyl_file or not os.path.isfile(cyl_file):
        if cyl_file:
            print(f"WARNING: {cyl_file} not found; using global membrane COM (z_cyl = 0).")
        return {}
    table = np.loadtxt(cyl_file, skiprows=1)
    if table.ndim == 1:
        table = table.reshape(1, -1)
    return {(int(f), int(r)): float(z) for f, r, z in table}


def process(psf_file, dcd_file, resname, cyl_file=None):
    u = mda.Universe(psf_file, dcd_file)

    # Cylinder-based local membrane COM reference (Nitschke 2016). z_solute is
    # measured relative to this local COM instead of the global all-lipid COM.
    cyl_com = load_cylinder_com(cyl_file)
    missing_cyl = 0

    # Select heavy atoms of the analog
    analog_heavy = u.select_atoms(f"resname {resname} and not name H*")
    residue_ids = sorted(set(analog_heavy.resids))
    n_residues = len(residue_ids)
    print(f"Loaded {len(u.trajectory)} frames, {n_residues} analog residues ({resname})")

    # Pre-build per-residue heavy-atom selections
    res_selections = {}
    for rid in residue_ids:
        res_selections[rid] = u.select_atoms(
            f"resname {resname} and not name H* and resid {rid}"
        )

    # Collect data: frame, resid, x, y, z, 6A_cutoff, 8A_cutoff, 10A_cutoff, 12A_cutoff
    records = []

    for ts in u.trajectory:
        frame = ts.frame
        box = ts.dimensions

        for rid in residue_ids:
            sel = res_selections[rid]
            com = sel.center_of_mass()

            # Local membrane COM z for this solute/frame (0.0 if unavailable).
            if cyl_com:
                if (frame, int(rid)) in cyl_com:
                    z_ref = cyl_com[(frame, int(rid))]
                else:
                    z_ref = 0.0
                    missing_cyl += 1
            else:
                z_ref = 0.0

            # Positions of this residue's heavy atoms
            pos_this = sel.positions
            # Positions of all OTHER residues' heavy atoms
            others = u.select_atoms(
                f"resname {resname} and not name H* and not resid {rid}"
            )
            pos_others = others.positions

            row = {
                "frame": frame,
                "resid": rid,
                "x": round(float(com[0]), 4),
                "y": round(float(com[1]), 4),
                "z": round(float(com[2] - z_ref), 4),  # cylinder-referenced reaction coordinate
            }

            if len(pos_others) > 0:
                # Distance matrix: (atoms_this x atoms_others)
                dist_matrix = distance_array(pos_this, pos_others, box=box)
                # Map other atoms back to their residue ids
                other_resids = others.resids

                for cutoff in cutoffs:
                    # For each other atom, check if any atom of this residue is within cutoff
                    min_per_other_atom = dist_matrix.min(axis=0)  # min distance from this residue
                    in_contact = min_per_other_atom < cutoff
                    # Count unique residues in contact
                    contacting_resids = set(other_resids[in_contact])
                    row[f"{cutoff:.0f}A_cutoff"] = len(contacting_resids)
            else:
                for cutoff in cutoffs:
                    row[f"{cutoff:.0f}A_cutoff"] = 0

            records.append(row)

        if frame % 50 == 0:
            print(f"  Frame {frame}/{len(u.trajectory) - 1} processed")

    if cyl_com and missing_cyl:
        print(f"WARNING: {missing_cyl} (frame, resid) pairs missing from the cylinder "
              f"table; those used z_cyl = 0.")
    print(f"Done. {len(records)} records collected.")
    return records

## 3. Build Polars DataFrame and save raw data

In [ ]:
def save(records, output_file):
    df = pl.DataFrame(records)
    print(df.head(20))
    print(f"\nShape: {df.shape}")

    # Save with same format as densityProfiles-aa-cutoffs.vmd output
    with open(output_file, "w") as f:
        f.write("frame\tresid\tx\ty\tz\t4.5A_cutoff\n")
        for row in df.iter_rows():
            f.write(f"{row[0]}\t{row[1]}\t{row[2]:.4f}\t{row[3]:.4f}\t{row[4]:.4f}\t{row[5]}\n")

    print(f"\nSaved {output_file} ({df.height} rows)")

In [ ]:
!mkdir -p contacts
for res in resname:
    print("Preprocessing...")
    psf_file, output_file = preprocess(res)
    !mkdir -p contacts/{res.lower()}
    for i in range(1,4):
        print("Processing...")
        dcd_file = f"trajectory{i}.dcd"
        cyl_file = f"cylinder_com{i}.dat"   # per-solute cylinder COM z written by center.vmd
        records = process(psf_file, dcd_file, res, cyl_file)
        save(records, output_file)
        !mv {output_file} contacts/{res.lower()}/{res.lower()}_contacts_{i}.dat
    !rm trajectory*.dcd cylinder_com*.dat temp* analog*
!mkdir -p ../../../figures/data/distribution_data/raw_data_cyl/
!mv contacts/* ../../../figures/data/distribution_data/raw_data_cyl/ && rmdir contacts